In [1]:
import os
import tarfile
import json
import io
from pathlib import Path

def create_clip_webdataset(root_dir, output_filename):
    root_path = Path(root_dir)
    # Get all ingredient folders (ignoring hidden files)
    ingredient_folders = [f for f in root_path.iterdir() if f.is_dir()]
    
    with tarfile.open(output_filename, "w") as tar:
        count = 0
        for folder in ingredient_folders:
            ingredient_name = folder.name
            # Path to the thumbs directory
            thumbs_path = folder / "thumbs"
            
            if not thumbs_path.exists():
                continue
                
            for img_path in thumbs_path.glob("*"):
                if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.webp']:
                    continue
                
                # Create a unique basename for this sample
                basename = f"{count:06d}"
                
                # 1. Add Image File
                tar.add(img_path, arcname=f"{basename}{img_path.suffix}")
                
                # 2. Add Metadata (JSON) for Prompt Ensembling
                # We store the raw ingredient and a few pre-generated prompts
                metadata = {
                    "ingredient": ingredient_name,
                    "prompts": [
                        f"a photo of {ingredient_name}",
                        f"a close-up shot of {ingredient_name}",
                        f"fresh {ingredient_name} for cooking",
                        f"an image of the ingredient {ingredient_name}",
                        f"a photo of raw {ingredient_name}",
                        f"{ingredient_name} on a white background",
                        f"fresh {ingredient_name} on a cutting board",
                        f"{ingredient_name} at a grocery store",
                        f"a high quality photo of {ingredient_name}"
                    ]
                }
                json_data = json.dumps(metadata).encode('utf-8')
                json_info = tarfile.TarInfo(name=f"{basename}.json")
                json_info.size = len(json_data)
                tar.addfile(json_info, io.BytesIO(json_data))
                
                count += 1
                if count % 100 == 0:
                    print(f"Packed {count} images...")

# Usage
create_clip_webdataset("./saved_pages", "ingredients_data.tar")

Packed 100 images...


## Using .tar file with CLIP and Prompt Ensembling

In [2]:
from google.colab import drive
drive.mount("/content/drive")

MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b0-3ut0le2iga2t2?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_lo

In [ ]:
import torch
import clip
import webdataset as wds
from PIL import Image
import io

# 1. Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# 2. Define the WebDataset Pipeline
dataset = (
    wds.WebDataset("ingredients_data.tar")
    .decode("pil") # Decodes images automatically
    .to_tuple("jpg;png;webp", "json") # Get image and the json metadata
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)

def ensemble_inference(image_pil, metadata):
    # Process Image
    image_input = preprocess(image_pil).unsqueeze(0).to(device)
    
    # Process Ensemble of Prompts
    prompts = metadata['prompts']
    text_inputs = clip.tokenize(prompts).to(device)
    
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_inputs)
        
        # Normalize features
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Average the text features (The Ensemble Step)
        # This creates a single "super-concept" vector for the ingredient
        mean_text_feature = text_features.mean(dim=0, keepdim=True)
        mean_text_feature /= mean_text_feature.norm(dim=-1, keepdim=True)
        
        # Calculate similarity
        similarity = (100.0 * image_features @ mean_text_feature.T)
        
    return similarity

# Example Loop
for img, meta in dataloader:
    # Note: WebDataset/CLIP handling usually requires custom batching logic 
    # for ensembles, but here is the logic for a single item:
    score = ensemble_inference(img[0], meta[0])
    print(f"Confidence score for {meta[0]['ingredient']}: {score.item():.2f}")
    break